In [32]:
import os
import copy
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms


In [33]:
BASE_DIR = r"C:\Users\SAKTHI\Desktop\SMARTVISION\smartvision_dataset"

TRAIN_DIR = f"{BASE_DIR}/classification/train"
VAL_DIR = f"{BASE_DIR}/classification/val"
TEST_DIR = f"{BASE_DIR}/classification/test"

os.makedirs("models", exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("✅ Setup complete.")
print("Device:", device)

✅ Setup complete.
Device: cuda


In [34]:
# Image Transforms
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std = [0.229, 0.224, 0.225]
    )
])

In [35]:
# Load Dataset
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

class_names = train_dataset.classes
num_classes = len(class_names)

print("\nClasses: ", class_names)
print("\n Classes length",num_classes )


Classes:  ['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted_plant', 'stop_sign', 'traffic_light', 'train', 'truck']

 Classes length 26


In [36]:
# LOAD PRETRAINED MODEL
# =====================================================

weights = models.MobileNet_V2_Weights.IMAGENET1K_V2
model = models.mobilenet_v2(weights=weights)

print(model.classifier)


Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=1000, bias=True)
)


In [37]:
for param in model.features.parameters():
    param.requires_grad = False

print("Convolutional base frozen.")


Convolutional base frozen.


In [38]:

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(1280 , 256),
    nn.ReLU(inplace=True),
    nn.Dropout(0.2),
    nn.Linear(256, 26),
)
model = model.to(device)


In [39]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001,
)

scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)


In [40]:
best_val_acc = 0.0
epochs_no_improve = 0
best_model_weights = None


In [41]:
BEST_WEIGHTS_PATH = 'models/mobilenet_best.pth'
best_val_acc = 0.0  # initialize before training loop

for epoch in range(1, 15 + 1):

    # ----------------- Training phase -----------------
    model.train()
    model.features.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_correct += (outputs.argmax(1) == labels).sum().item()
        total += inputs.size(0)

    train_loss = running_loss / total
    train_acc = running_correct / total

    # ----------------- Validation phase -----------------
    model.eval()
    val_running_loss = 0.0
    val_running_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * inputs.size(0)
            val_running_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += inputs.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_running_correct / val_total

    # ----------------- Scheduler step -----------------
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch {epoch:02d}/{15} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.2e}"
    )

    # ----------------- Early stopping + save-best check -----------------
    if val_acc > best_val_acc + 1e-4:
        best_val_acc = val_acc
        epochs_no_improve = 0
        best_model_weights = copy.deepcopy(model.state_dict())

        # Save best weights to disk immediately, so we always have the
        # latest "best" version even if training is interrupted.
        torch.save(best_model_weights, BEST_WEIGHTS_PATH)
        print(f"  -> New best val_acc={val_acc:.4f}. Weights saved to '{BEST_WEIGHTS_PATH}'.")
    else:
        epochs_no_improve += 1
        EARLY_STOPPING_PATIENCE = 7
        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print(f"\nNo improvement for {EARLY_STOPPING_PATIENCE} epochs "
                  f"-> stopping early at epoch {epoch}.")
            break

Epoch 01/15 | train_loss=2.0265 train_acc=0.4929 | val_loss=1.0766 val_acc=0.6897 | lr=1.00e-03
  -> New best val_acc=0.6897. Weights saved to 'models/mobilenet_best.pth'.
Epoch 02/15 | train_loss=0.8595 train_acc=0.7440 | val_loss=0.9253 val_acc=0.7026 | lr=1.00e-03
  -> New best val_acc=0.7026. Weights saved to 'models/mobilenet_best.pth'.
Epoch 03/15 | train_loss=0.6299 train_acc=0.7967 | val_loss=0.8497 val_acc=0.7308 | lr=1.00e-03
  -> New best val_acc=0.7308. Weights saved to 'models/mobilenet_best.pth'.
Epoch 04/15 | train_loss=0.4800 train_acc=0.8643 | val_loss=0.8760 val_acc=0.7154 | lr=1.00e-03
Epoch 05/15 | train_loss=0.3605 train_acc=0.9027 | val_loss=0.8347 val_acc=0.7205 | lr=1.00e-03
Epoch 06/15 | train_loss=0.2845 train_acc=0.9159 | val_loss=0.8849 val_acc=0.7385 | lr=1.00e-03
  -> New best val_acc=0.7385. Weights saved to 'models/mobilenet_best.pth'.
Epoch 07/15 | train_loss=0.2315 train_acc=0.9341 | val_loss=0.8792 val_acc=0.7282 | lr=1.00e-03
Epoch 08/15 | train_loss

In [42]:
print(f"Best model weights loaded. Best val_loss = {val_acc:.4f}")
print(f"Saved at: {BEST_WEIGHTS_PATH}")


Best model weights loaded. Best val_loss = 0.7385
Saved at: models/mobilenet_best.pth
